# Radar de Riesgo Social — Setup de la API en Colab

Notebook listo para ejecutar **celda por celda, en orden**. No hay que modificar nada
salvo **una sola cosa**: pegar tu token de ngrok en la celda 5 (marcado como `TU_TOKEN_AQUI`).

**Antes de empezar:** activá GPU en `Entorno de ejecución → Cambiar tipo de entorno → GPU`.
El pipeline usa el modelo NLI `mDeBERTa-v3` y sin GPU es inusable.

Orden de las celdas:
1. Clonar el repo y cambiar a la rama correcta
2. Instalar dependencias
3. Pre-descargar el modelo NLI
4. Arrancar la API con logs visibles
5. Servir el front y exponer con ngrok


## 1. Clonar el repo y cambiar a la rama


In [ ]:
%cd /content
![ -d webscrapping_transformers ] && rm -rf webscrapping_transformers
!git clone https://github.com/botanicalex/webscrapping_transformers.git
%cd /content/webscrapping_transformers
!git checkout integracion-front-back
!git log --oneline -3


## 2. Instalar dependencias

Instala en un **orden específico** para esquivar el conflicto torch/torchvision/numpy de
Colab: **no reinstala torch** (usa el de Colab), agrega `torchvision` del índice `cu128`,
`transformers`/`tokenizers` **sin tocar sus deps** (`--no-deps`, así no mueve numpy), y el
resto (API + scraping). **Después de esta celda hay que reiniciar el runtime** (ver la
celda siguiente).

In [ ]:
%cd /content/webscrapping_transformers

# Orden EXACTO para esquivar el conflicto torch/torchvision/numpy de Colab.
# NO se reinstala torch: se usa el que ya trae Colab (2.8.0+cu128).
!pip install torchvision --index-url https://download.pytorch.org/whl/cu128
!pip install transformers==4.57.1 tokenizers --no-deps
!pip install fastapi uvicorn httpx pandas openpyxl scikit-learn pyngrok newspaper3k lxml_html_clean playwright aiohttp nest-asyncio
!playwright install chromium
print('Dependencias instaladas. Reinicia el runtime antes de continuar (ver celda siguiente).')

## ⚠️ IMPORTANTE: Reinicia el runtime ahora

**Reinicia el runtime ahora (`Runtime → Restart session`) antes de continuar.**

Es necesario para que Python recargue torch / torchvision / numpy con las versiones recién
instaladas. Tras el reinicio, continuá **desde la celda 3** (no repitas las celdas 1 y 2;
lo instalado persiste).

## 3. Pre-descargar el modelo NLI

Verifica primero que `torchvision` importa sin el error de `nms`, y deja `mDeBERTa-v3` en el
caché de Hugging Face para que el arranque de la API (celda 4) no tenga que descargarlo.
Correr **después de reiniciar el runtime**.

In [ ]:
import torchvision  # verificar que importa bien
print("torchvision OK:", torchvision.__version__)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODELO_NLI = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
AutoTokenizer.from_pretrained(MODELO_NLI)
AutoModelForSequenceClassification.from_pretrained(MODELO_NLI)
print("✅ Modelo en caché")

## 4. Arrancar la API con logs visibles

Levanta `uvicorn src.api:app` en segundo plano (logs en `api.log`) y **espera a que
`/health` responda**. La API carga el modelo NLI al arrancar, así que esto puede tardar
**varios minutos**. La celda va imprimiendo las últimas líneas del log mientras espera.


In [ ]:
import subprocess, time, requests

# Cierra una API previa si quedó corriendo
try:
    api.terminate()
except Exception:
    pass

log = open('api.log', 'w')
api = subprocess.Popen(
    ['uvicorn', 'src.api:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT,
)
print('uvicorn PID:', api.pid, '\nEsperando a que cargue el modelo y responda /health...\n')

inicio = time.time()
lista = False
while time.time() - inicio < 900:  # hasta 15 min
    time.sleep(5)
    with open('api.log') as f:
        cola = f.readlines()[-3:]
    if cola:
        print('  ' + '  '.join(l for l in cola).strip())
    if api.poll() is not None:
        print('\n>> uvicorn se detuvo. Log completo:\n')
        print(open('api.log').read())
        break
    try:
        r = requests.get('http://localhost:8000/health', timeout=3)
        if r.ok:
            print('\n>> API lista:', r.json())
            lista = True
            break
    except Exception:
        pass

if not lista and api.poll() is None:
    print('\n>> Timeout esperando /health (15 min). Revisá api.log.')


## 5. Servir el front y exponer con ngrok

Abre un túnel HTTPS público a la API (puerto 8000) y arma el **link del front** ya
conectado a ese túnel vía el parámetro `?api=`.

> **PEGA TU TOKEN DE NGROK** en `NGROK_TOKEN` abajo (lo sacás de
> https://dashboard.ngrok.com/get-started/your-authtoken). Es lo **único** que hay que editar.


In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok

NGROK_TOKEN = "TU_TOKEN_AQUI"   # <-- PEGA TU TOKEN DE NGROK AQUI
ngrok.set_auth_token(NGROK_TOKEN)

# Cierra túneles previos (ngrok free permite uno a la vez)
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

api_url = ngrok.connect(8000, 'http').public_url
if api_url.startswith('http://'):
    api_url = 'https://' + api_url[len('http://'):]

FRONT = 'https://botanicalex.github.io/webscrapping_transformers/busqueda_pipeline.html'

print('API publica :', api_url)
print('Health      :', api_url + '/health')
print('\nAbri el front ya conectado a esta API:\n')
print(f'{FRONT}?api={api_url}')


---
### Utilidades

Ver las últimas líneas del log de la API (por si un `/analizar` falla):


In [ ]:
!tail -n 40 api.log


Detener todo (API + túnel) cuando termines:


In [ ]:
try:
    from pyngrok import ngrok
    ngrok.kill()
    print('Túnel ngrok cerrado.')
except Exception as e:
    print('ngrok:', e)
try:
    api.terminate()
    print('uvicorn detenido.')
except Exception as e:
    print('uvicorn:', e)
